# RAG Prototype 1 — Ingestion & Chunking

Imports → configuration → data inventory → document extraction → ingestion run → structuring → chunking.

Cleanup pass on this notebook: data acquisition is dropped (handled upstream — files just need to land in
`data/raw/`), and every function below keeps only its final, working version — the earlier iterations that
were superseded while building `detect_docx_structure`, `build_docx_sections`, and `chunk_structural_unit`
have been removed rather than left stacked on top of each other. One real bug got fixed along the way — see
the note in the Chunking section.

## 1. Imports

In [1]:
import json
import logging
import re
from collections import Counter
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

import pandas as pd
import pymupdf
from docx import Document
from openpyxl import load_workbook

## 2. Configuration & Logging

`data/` is split into `data/raw/` (exactly what was handed to you — never modified) and `data/processed/`
(what ingestion produces — safe to delete and regenerate at any time). Every tunable number used later
(thresholds for "is this column an id?", "is this line a repeated header?", etc.) lives here too, instead of
being buried inside the function that uses it.

In [2]:
@dataclass
class Config:
    base_dir: Path = field(default_factory=Path.cwd)
    data_subdir: str = "data"
    raw_subdir: str = "raw"                # data/raw       — untouched source files
    processed_subdir: str = "processed"    # data/processed — ingestion output (regenerable)
    output_subdir: str = "outputs"
    log_subdir: str = "logs"

    supported_extensions: tuple = (".pdf", ".docx", ".xlsx")

    short_page_word_threshold: int = 20

    # A line that repeats across at least this fraction of a PDF's pages is
    # treated as a running header/footer and stripped.
    pdf_boilerplate_min_repeat_ratio: float = 0.4

    # An xlsx column whose non-empty values average fewer words than this
    # is treated as an identifier/category rather than free text.
    xlsx_semantic_min_avg_words: float = 3.0
    xlsx_id_like_names: frozenset = frozenset({
        "id", "row_id", "index", "rank", "ranking", "serial", "sr_no", "s_no",
    })

    @property
    def data_dir(self) -> Path:
        return self.base_dir / self.data_subdir

    @property
    def raw_dir(self) -> Path:
        return self.data_dir / self.raw_subdir

    @property
    def processed_dir(self) -> Path:
        return self.data_dir / self.processed_subdir

    @property
    def output_dir(self) -> Path:
        return self.base_dir / self.output_subdir

    @property
    def log_dir(self) -> Path:
        return self.base_dir / self.log_subdir

    def ensure_dirs(self):
        for directory in (self.raw_dir, self.processed_dir, self.output_dir, self.log_dir):
            directory.mkdir(exist_ok=True, parents=True)


CONFIG = Config()
CONFIG.ensure_dirs()

print(f"Raw directory:       {CONFIG.raw_dir}")
print(f"Processed directory: {CONFIG.processed_dir}")
print(f"Output directory:    {CONFIG.output_dir}")
print(f"Log directory:       {CONFIG.log_dir}")

Raw directory:       c:\Users\Swapn\OneDrive\Documents\GitHub\Surfprice\data\raw
Processed directory: c:\Users\Swapn\OneDrive\Documents\GitHub\Surfprice\data\processed
Output directory:    c:\Users\Swapn\OneDrive\Documents\GitHub\Surfprice\outputs
Log directory:       c:\Users\Swapn\OneDrive\Documents\GitHub\Surfprice\logs


**One-time manual step:** make sure your source files (`Attention_Is_All_You_Need.docx`,
`Foundation-LLMs.pdf`, `rag.xlsx`, the RAG paper `.docx`) are sitting in `data/raw/`. Everything from this
point on reads from `raw_dir` and writes to `processed_dir`.

In [3]:
logger = logging.getLogger("rag_ingestion")
logger.setLevel(logging.INFO)
logger.handlers.clear()  # avoid duplicate handlers if this cell is re-run

_formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")

_console_handler = logging.StreamHandler()
_console_handler.setFormatter(_formatter)
logger.addHandler(_console_handler)

_file_handler = logging.FileHandler(CONFIG.log_dir / "ingestion.log")
_file_handler.setFormatter(_formatter)
logger.addHandler(_file_handler)

logger.info("Logger initialized.")

2026-08-27 11:39:05,399 | INFO | Logger initialized.


## 3. Data Inventory

In [4]:
def list_data_files(config: Config = CONFIG) -> list[Path]:
    files = sorted(config.raw_dir.iterdir())

    logger.info(f"Files in raw directory: {len(files)}")
    for file in files:
        size_kb = file.stat().st_size / 1024
        logger.info(f"- {file.name} | {file.suffix or 'no ext'} | {size_kb:.2f} KB")

    return files


data_files = list_data_files()

2026-08-27 11:39:08,895 | INFO | Files in raw directory: 4
2026-08-27 11:39:08,897 | INFO | - Attention_Is_All_You_Need.docx | .docx | 330.47 KB
2026-08-27 11:39:08,897 | INFO | - Foundation-LLMs.pdf | .pdf | 2656.22 KB
2026-08-27 11:39:08,900 | INFO | - rag.xlsx | .xlsx | 898.20 KB
2026-08-27 11:39:08,901 | INFO | - Retrieval-Augmented_Generation_for_Knowledge-Intensive_NLP_Tasks.docx | .docx | 245.31 KB


## 4. Document Extraction

### 4.1 Shared helpers

`clean_text()` and `compute_stats()` run for every record regardless of source type, so every record ends up
with the same cleaned `text` and the same `char_count` / `word_count` / `line_count` fields.

In [5]:
def clean_text(text: str) -> str:
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def compute_stats(text: str) -> dict:
    return {
        "char_count": len(text),
        "word_count": len(text.split()),
        "line_count": len(text.splitlines()),
    }


def make_record(document: str, source_type: str, location: str, text: str, metadata: dict | None = None) -> dict:
    cleaned = clean_text(text)
    record = {
        "document": document,
        "source_type": source_type,
        "location": location,
        "text": cleaned,
        "metadata": metadata or {},
    }
    record.update(compute_stats(cleaned))
    return record

### 4.2 PDF — with running header/footer removal

A line that shows up on a large fraction of a document's pages (a running header, a page footer, a copyright
notice) carries no content and just adds noise to every chunk. `find_boilerplate_lines()` finds those lines
per-document, and `extract_pdf` strips them before the text is stored.

In [6]:
def find_boilerplate_lines(page_texts: list[str], config: Config = CONFIG) -> set[str]:
    """Lines that repeat across a large fraction of a document's pages — treated as
    running headers/footers rather than content."""
    if len(page_texts) < 3:
        return set()  # too few pages for repetition to mean anything

    line_counts = Counter()
    for text in page_texts:
        unique_lines_on_page = {line.strip() for line in text.splitlines() if line.strip()}
        line_counts.update(unique_lines_on_page)

    threshold = max(2, int(len(page_texts) * config.pdf_boilerplate_min_repeat_ratio))
    return {line for line, count in line_counts.items() if count >= threshold}


def remove_lines(text: str, lines_to_remove: set[str]) -> str:
    if not lines_to_remove:
        return text
    kept = [line for line in text.splitlines() if line.strip() not in lines_to_remove]
    return "\n".join(kept)


def extract_pdf(file_path: Path, config: Config = CONFIG) -> list[dict]:
    with pymupdf.open(file_path) as doc:
        page_texts = [page.get_text() for page in doc]

    boilerplate = find_boilerplate_lines(page_texts, config)
    if boilerplate:
        logger.info(f"{file_path.name}: stripping {len(boilerplate)} repeated header/footer line(s)")

    documents = []
    for page_number, text in enumerate(page_texts, start=1):
        text = remove_lines(text, boilerplate)
        documents.append(
            make_record(
                document=file_path.name,
                source_type="pdf",
                location=f"page_{page_number}",
                text=text,
            )
        )

    return documents

### 4.3 DOCX

The paragraph's Word style (`Heading 1`, `Normal`, etc.) is kept in `metadata` — cheap to capture now, and
used later for telling headings apart from body text when structuring.

In [7]:
def extract_docx(file_path: Path) -> list[dict]:
    documents = []

    doc = Document(file_path)

    for index, paragraph in enumerate(doc.paragraphs, start=1):
        text = paragraph.text.strip()

        if text:
            documents.append(
                make_record(
                    document=file_path.name,
                    source_type="docx",
                    location=f"paragraph_{index}",
                    text=text,
                    metadata={"style": paragraph.style.name},
                )
            )

    return documents

### 4.4 XLSX — column classification (simplified)

Not every column in a spreadsheet is meant to be embedded as text. An `id` column or a short category column
just adds noise if it's mashed into the same string as the real content. Two signals decide the split:

- **name** — does the header look like `id`, `index`, `rank`, etc.?
- **average word count** — free text (`question`, `answer`) reads as multiple words per cell; identifiers
  and short categories read as one or two.

`semantic` columns are joined into `text` (what actually gets embedded later). `metadata` columns are
attached to `metadata` on the record instead of thrown away — still there if you need to filter or trace a
chunk back to its row.

In [8]:
def classify_xlsx_columns(headers: list[str], data_rows: list[tuple], config: Config = CONFIG) -> dict[str, str]:
    columns = {header: [] for header in headers}
    for row in data_rows:
        for header, value in zip(headers, row):
            columns[header].append(value)

    classification = {}
    for header, values in columns.items():
        non_empty = [str(v).strip() for v in values if v is not None and str(v).strip()]
        avg_word_count = (
            sum(len(v.split()) for v in non_empty) / len(non_empty)
            if non_empty else 0
        )

        looks_like_id = header.strip().lower() in config.xlsx_id_like_names
        looks_short = avg_word_count < config.xlsx_semantic_min_avg_words

        classification[header] = "metadata" if (looks_like_id or looks_short) else "semantic"

    return classification

Preview the classification before running full ingestion — worth a glance so a real free-text column doesn't quietly get misclassified as metadata.

In [9]:
def read_xlsx_rows(file_path: Path) -> list[tuple[str, list[str], list[tuple]]]:
    workbook = load_workbook(file_path, read_only=True, data_only=True)
    sheets = []

    for sheet in workbook.worksheets:
        rows = list(sheet.iter_rows(values_only=True))
        if not rows:
            continue

        headers = [
            str(value).strip() if value is not None else f"column_{i}"
            for i, value in enumerate(rows[0])
        ]
        sheets.append((sheet.title, headers, rows[1:]))

    workbook.close()
    return sheets


column_preview = []
for file_path in CONFIG.raw_dir.glob("*.xlsx"):
    for sheet_title, headers, data_rows in read_xlsx_rows(file_path):
        roles = classify_xlsx_columns(headers, data_rows)
        for header, role in roles.items():
            column_preview.append({
                "file": file_path.name,
                "sheet": sheet_title,
                "column": header,
                "role": role,
            })

pd.DataFrame(column_preview)

,file,sheet,column,role
0,rag.xlsx,Sheet1,question,semantic
1,rag.xlsx,Sheet1,answer,semantic
2,rag.xlsx,Sheet1,relevant_passage_ids,semantic
3,rag.xlsx,Sheet1,id,metadata


In [10]:
def extract_xlsx(file_path: Path, config: Config = CONFIG) -> list[dict]:
    documents = []

    for sheet_title, headers, data_rows in read_xlsx_rows(file_path):
        column_roles = classify_xlsx_columns(headers, data_rows, config)
        semantic_headers = [h for h in headers if column_roles[h] == "semantic"]
        metadata_headers = [h for h in headers if column_roles[h] == "metadata"]

        logger.info(
            f"{file_path.name} [{sheet_title}]: semantic={semantic_headers} metadata={metadata_headers}"
        )

        for row_number, row in enumerate(data_rows, start=2):  # row 1 was the header
            row_dict = dict(zip(headers, row))

            semantic_values = [
                str(row_dict[h]).strip()
                for h in semantic_headers
                if row_dict.get(h) is not None and str(row_dict[h]).strip()
            ]
            if not semantic_values:
                continue

            text = " | ".join(semantic_values)

            row_metadata = {"sheet": sheet_title, "row": row_number}
            for h in metadata_headers:
                if row_dict.get(h) is not None:
                    row_metadata[h] = row_dict[h]

            documents.append(
                make_record(
                    document=file_path.name,
                    source_type="xlsx",
                    location=f"{sheet_title}!row_{row_number}",
                    text=text,
                    metadata=row_metadata,
                )
            )

    return documents


EXTRACTORS = {
    ".pdf": extract_pdf,
    ".docx": extract_docx,
    ".xlsx": extract_xlsx,
}


def ingest_file(file_path: Path) -> list[dict]:
    suffix = file_path.suffix.lower()

    extractor = EXTRACTORS.get(suffix)
    if extractor is None:
        raise ValueError(f"Unsupported file format: {suffix}")

    return extractor(file_path)

## 5. Run Ingestion

Reads from `raw_dir`, wraps each file in its own `try/except`, and saves the final record list to
`processed_dir` as JSON — so this notebook (or a later one) can just load that file instead of re-running
extraction.

In [11]:
def run_ingestion(config: Config = CONFIG) -> list[dict]:
    all_documents = []

    for file_path in sorted(config.raw_dir.iterdir()):
        if file_path.suffix.lower() not in config.supported_extensions:
            continue

        try:
            extracted = ingest_file(file_path)
            all_documents.extend(extracted)
            logger.info(f"{file_path.name}: {len(extracted)} records")

        except Exception:
            logger.exception(f"Failed to ingest {file_path.name}, skipping.")

    logger.info(f"Total records: {len(all_documents)}")
    return all_documents


def save_documents(documents: list[dict], config: Config = CONFIG, filename: str = "ingested_records.json") -> Path:
    out_path = config.processed_dir / filename
    out_path.write_text(json.dumps(documents, indent=2, ensure_ascii=False), encoding="utf-8")
    logger.info(f"Saved {len(documents)} records to {out_path}")
    return out_path


documents = run_ingestion()
save_documents(documents)

2026-08-27 11:39:49,422 | INFO | Attention_Is_All_You_Need.docx: 298 records
2026-08-27 11:39:50,468 | INFO | Foundation-LLMs.pdf: 277 records
2026-08-27 11:39:51,136 | INFO | rag.xlsx [Sheet1]: semantic=['question', 'answer', 'relevant_passage_ids'] metadata=['id']
2026-08-27 11:39:51,363 | INFO | rag.xlsx: 4719 records
2026-08-27 11:39:51,856 | INFO | Retrieval-Augmented_Generation_for_Knowledge-Intensive_NLP_Tasks.docx: 211 records
2026-08-27 11:39:51,860 | INFO | Total records: 5505
2026-08-27 11:39:52,029 | INFO | Saved 5505 records to c:\Users\Swapn\OneDrive\Documents\GitHub\Surfprice\data\processed\ingested_records.json


WindowsPath('c:/Users/Swapn/OneDrive/Documents/GitHub/Surfprice/data/processed/ingested_records.json')

### Sanity checks

In [12]:
empty_docs = [doc for doc in documents if not doc["text"]]
short_docs = [
    doc for doc in documents
    if doc["word_count"] < CONFIG.short_page_word_threshold
]

print(f"Total records: {len(documents)}")
print(f"Empty records: {len(empty_docs)}")
print(f"Non-empty records: {len(documents) - len(empty_docs)}")
print(f"Suspiciously short records (< {CONFIG.short_page_word_threshold} words): {len(short_docs)}")

for doc in short_docs[:10]:
    print(doc["document"], "|", doc["location"], "| words:", doc["word_count"])

Total records: 5505
Empty records: 0
Non-empty records: 5505
Suspiciously short records (< 20 words): 619
Attention_Is_All_You_Need.docx | paragraph_3 | words: 5
Attention_Is_All_You_Need.docx | paragraph_10 | words: 5
Attention_Is_All_You_Need.docx | paragraph_11 | words: 5
Attention_Is_All_You_Need.docx | paragraph_12 | words: 5
Attention_Is_All_You_Need.docx | paragraph_13 | words: 5
Attention_Is_All_You_Need.docx | paragraph_17 | words: 5
Attention_Is_All_You_Need.docx | paragraph_18 | words: 8
Attention_Is_All_You_Need.docx | paragraph_19 | words: 2
Attention_Is_All_You_Need.docx | paragraph_20 | words: 2
Attention_Is_All_You_Need.docx | paragraph_21 | words: 1


## 6. Structuring — grouping DOCX records into sections

DOCX is the only format with a native heading signal (paragraph style), so it's the only one structured into
sections so far — PDF and XLSX records go straight into chunking as flat records (see the note at the end of
this notebook).

Heading detection went through a few iterations while building this — style-based only, then a
dict-key-mismatch version, then finally style **plus** a numbered-heading fallback (`"3.2 Related Work"`
with no `Heading` style still gets recognized). Only that final combined version is kept below, along with
the one `build_docx_sections` that uses it.

In [13]:
@dataclass
class StructuralUnit:
    document: str
    source_type: str

    title: str | None
    level: int | None

    text: str

    metadata: dict[str, Any] = field(default_factory=dict)

In [14]:
def detect_docx_structure(record: dict) -> int | None:
    """Heading level for a record, or None if it's body text.

    Primary signal: the DOCX paragraph style (Heading 1, Heading 2, ...).
    Fallback signal: a numbered heading in the text itself (e.g. "3.2 Related Work"),
    for documents where headings weren't styled consistently.
    """
    style = record.get("metadata", {}).get("style", "")

    if style == "Heading 1":
        return 1

    if style == "Heading 2":
        return 2

    text = record.get("text", "").strip()
    match = re.match(r"^(\d+(?:\.\d+)*)\s+.+$", text)

    if match:
        return match.group(1).count(".") + 1

    return None


def build_docx_sections(records: list[dict]) -> list[dict]:
    sections = []

    current_title = None
    current_level = None
    current_text = []
    section_path = []

    for record in records:
        text = record.get("text", "").strip()

        if not text:
            continue

        level = detect_docx_structure(record)

        if level is not None:
            if current_text:
                sections.append({
                    "document": record["document"],
                    "source_type": "docx",
                    "title": current_title,
                    "level": current_level,
                    "text": "\n".join(current_text),
                    "metadata": {"section_path": section_path.copy()},
                })

            current_title = text
            current_level = level

            section_path = section_path[:level - 1]
            section_path.append(text)

            current_text = []
        else:
            current_text.append(text)

    if current_text:
        sections.append({
            "document": records[0]["document"],
            "source_type": "docx",
            "title": current_title,
            "level": current_level,
            "text": "\n".join(current_text),
            "metadata": {"section_path": section_path.copy()},
        })

    return sections


def sections_to_structural_units(sections: list[dict]) -> list[StructuralUnit]:
    return [
        StructuralUnit(
            document=section["document"],
            source_type=section["source_type"],
            title=section["title"],
            level=section["level"],
            text=section["text"],
            metadata=section["metadata"],
        )
        for section in sections
    ]

In [15]:
docx_records = [d for d in documents if d["source_type"] == "docx"]

all_docx_units = []
for document in {d["document"] for d in docx_records}:
    records = [d for d in docx_records if d["document"] == document]
    sections = build_docx_sections(records)
    all_docx_units.extend(sections_to_structural_units(sections))

print("DOCX records:", len(docx_records))
print("DOCX structural units:", len(all_docx_units))
print()
print(all_docx_units[0])

DOCX records: 509
DOCX structural units: 44

StructuralUnit(document='Retrieval-Augmented_Generation_for_Knowledge-Intensive_NLP_Tasks.docx', source_type='docx', title=None, level=None, text='Retrieval-Augmented Generation for\nKnowledge-Intensive NLP Tasks\nPatrick Lewis†‡, Ethan Perez⋆,\nseq2seq baseline.', metadata={'section_path': []})


## 7. Chunking

**Bug fixed here:** the sliding-window chunker's whitespace-boundary step had a broken overlap calculation —
`start` was reassigned *outside* the `while` loop instead of inside it, so once a unit needed more than one
chunk, `start` never advanced and the loop would spin forever (it was never actually run to completion in
the original notebook — execution stopped right at this point). The version below computes the next
`start` inside the loop and always makes forward progress.

In [16]:
@dataclass
class Chunk:
    chunk_id: str

    document: str
    source_type: str

    text: str

    metadata: dict[str, Any] = field(default_factory=dict)

In [17]:
def chunk_structural_unit(unit: StructuralUnit, chunk_size: int = 1500, overlap: int = 200) -> list[dict]:
    text = unit.text.strip()

    if not text:
        return []

    if len(text) <= chunk_size:
        return [{
            "document": unit.document,
            "source_type": unit.source_type,
            "text": text,
            "metadata": {
                **unit.metadata,
                "section_title": unit.title,
                "section_level": unit.level,
                "chunk_index": 0,
            },
        }]

    chunks = []
    start = 0
    chunk_index = 0

    while start < len(text):
        end = min(start + chunk_size, len(text))

        # Back up to a whitespace boundary so we don't split mid-word
        if end < len(text):
            boundary = text.rfind(" ", start, end)
            if boundary > start:
                end = boundary

        chunk_text = text[start:end].strip()

        if chunk_text:
            chunks.append({
                "document": unit.document,
                "source_type": unit.source_type,
                "text": chunk_text,
                "metadata": {
                    **unit.metadata,
                    "section_title": unit.title,
                    "section_level": unit.level,
                    "chunk_index": chunk_index,
                },
            })
            chunk_index += 1

        if end >= len(text):
            break

        # Step forward by (chunk_size - overlap), snapped to a whitespace
        # boundary, and always past the current `start` so the loop can't stall.
        next_start = max(end - overlap, start + 1)
        boundary = text.find(" ", next_start, end)
        start = boundary + 1 if boundary != -1 else next_start

    return chunks


def chunk_dicts_to_objects(chunk_dicts: list[dict], unit_index: int = 0) -> list[Chunk]:
    chunks = []

    for chunk_index, item in enumerate(chunk_dicts):
        chunk_id = f"{item['document']}::unit_{unit_index}::chunk_{chunk_index}"

        chunks.append(
            Chunk(
                chunk_id=chunk_id,
                document=item["document"],
                source_type=item["source_type"],
                text=item["text"],
                metadata=item["metadata"],
            )
        )

    return chunks

In [18]:
all_docx_chunks = []

for unit_index, unit in enumerate(all_docx_units):
    chunk_dicts = chunk_structural_unit(unit)
    all_docx_chunks.extend(chunk_dicts_to_objects(chunk_dicts, unit_index=unit_index))

print("DOCX records:", len(docx_records))
print("Structural units:", len(all_docx_units))
print("Final chunks:", len(all_docx_chunks))

DOCX records: 509
Structural units: 44
Final chunks: 87


### Chunk-quality checks

In [19]:
chunk_lengths = [len(chunk.text) for chunk in all_docx_chunks]

oversized = [chunk for chunk in all_docx_chunks if len(chunk.text) > 1500]

missing_provenance = [
    chunk for chunk in all_docx_chunks
    if not chunk.document
    or not chunk.source_type
    or "section_path" not in chunk.metadata
]

print("Min chunk length:", min(chunk_lengths))
print("Max chunk length:", max(chunk_lengths))
print("Average chunk length:", sum(chunk_lengths) / len(chunk_lengths))
print("Oversized chunks (> 1500 chars):", len(oversized))
print("Chunks missing provenance:", len(missing_provenance))

for chunk in oversized[:10]:
    print(len(chunk.text), chunk.metadata.get("section_title"))

Min chunk length: 58
Max chunk length: 1499
Average chunk length: 1116.7816091954023
Oversized chunks (> 1500 chars): 0
Chunks missing provenance: 0


## 8. Next up

PDF and XLSX records don't have a heading signal, so they haven't been run through
`build_docx_sections`/`chunk_structural_unit` yet — that logic needs a PDF- and XLSX-appropriate equivalent
(or a shared fallback that just does fixed-size chunking directly on the flat record, no section step). The
cell below is a starting point: a look at what the raw PDF records currently look like.

In [20]:
pdf_records = [d for d in documents if d["source_type"] == "pdf"]

pdf_metadata_keys = Counter()
for record in pdf_records:
    pdf_metadata_keys.update(record.get("metadata", {}).keys())

location_patterns = Counter(record["location"] for record in pdf_records)

print("PDF records:", len(pdf_records))
print()
print("PDF metadata keys:")
for key, count in pdf_metadata_keys.most_common():
    print(f"  {key}: {count}")

print()
print("PDF locations (top 20):")
for location, count in location_patterns.most_common(20):
    print(f"  {location}: {count}")

print()
for record in pdf_records[:5]:
    print("=" * 80)
    print("Document:", record["document"])
    print("Location:", record["location"])
    print("Text:", record["text"][:200])

PDF records: 277

PDF metadata keys:

PDF locations (top 20):
  page_1: 1
  page_2: 1
  page_3: 1
  page_4: 1
  page_5: 1
  page_6: 1
  page_7: 1
  page_8: 1
  page_9: 1
  page_10: 1
  page_11: 1
  page_12: 1
  page_13: 1
  page_14: 1
  page_15: 1
  page_16: 1
  page_17: 1
  page_18: 1
  page_19: 1
  page_20: 1

Document: Foundation-LLMs.pdf
Location: page_1
Text: arXiv:2501.09223v2 [cs.CL] 15 Jun 2025
Foundations of
Large Language Models
Tong Xiao and Jingbo Zhu
June 17, 2025
NLP Lab, Northeastern University & NiuTrans Research
This book is a selection of chap
Document: Foundation-LLMs.pdf
Location: page_2
Text: Copyright © 2021-2025 Tong Xiao and Jingbo Zhu
NATURAL LANGUAGE PROCESSING LAB, NORTHEASTERN UNIVERSITY
&
NIUTRANS RESEARCH
Licensed under the Creative Commons Attribution-NonCommercial 4.0 Unported L
Document: Foundation-LLMs.pdf
Location: page_3
Text: Preface
Large language models originated from natural language processing, but they have undoubtedly
become one of the most r

In [21]:
# --- PDF structure detection: font-size (primary) + numbered-heading (fallback) ---
# Add to Config:
#     pdf_heading_size_ratio_h1: float = 1.4   # line font_size / body font_size
#     pdf_heading_size_ratio_h2: float = 1.15

def extract_pdf_lines(file_path: Path, config: Config = CONFIG) -> list[dict]:
    """Line-level PDF records with font metadata — needed for heading detection,
    which extract_pdf's page-level text doesn't preserve."""
    with pymupdf.open(file_path) as doc:
        boilerplate = find_boilerplate_lines([page.get_text() for page in doc], config)

        lines = []
        for page_number, page in enumerate(doc, start=1):
            for block in page.get_text("dict")["blocks"]:
                for line in block.get("lines", []):
                    spans = line.get("spans", [])
                    if not spans:
                        continue

                    text = clean_text("".join(s["text"] for s in spans))
                    if not text or text in boilerplate:
                        continue

                    lines.append({
                        "document": file_path.name,
                        "page": page_number,
                        "text": text,
                        "font_size": round(max(s["size"] for s in spans), 1),
                        "is_bold": any("bold" in s["font"].lower() for s in spans),
                    })

    return lines


def compute_body_font_size(lines: list[dict]) -> float:
    """The most common font size in the document — treated as the body-text baseline
    that headings are measured against."""
    if not lines:
        return 0
    return Counter(l["font_size"] for l in lines).most_common(1)[0][0]


def detect_pdf_heading_level(line: dict, body_font_size: float, config: Config = CONFIG) -> int | None:
    """Heading level for a PDF line, or None if it's body text.

    Primary signal: font size relative to the body-text baseline (bigger => higher-level
    heading — same idea as DOCX's Heading 1/2 styles, just measured instead of named).
    Fallback signal: a numbered heading (e.g. "3.2 Related Work") even at body size,
    same fallback used for DOCX.
    """
    size_ratio = line["font_size"] / body_font_size if body_font_size else 1.0

    if size_ratio >= config.pdf_heading_size_ratio_h1:
        return 1
    if size_ratio >= config.pdf_heading_size_ratio_h2:
        return 2

    match = re.match(r"^(\d+(?:\.\d+)*)\s+.+$", line["text"])
    if match and (line["is_bold"] or size_ratio >= 1.0):
        return match.group(1).count(".") + 1

    return None


def build_pdf_sections(lines: list[dict], body_font_size: float) -> list[dict]:
    """Same grouping logic as build_docx_sections, driven by detect_pdf_heading_level
    instead of detect_docx_structure."""
    sections = []
    current_title = None
    current_level = None
    current_text = []
    section_path = []

    for line in lines:
        text = line["text"]
        level = detect_pdf_heading_level(line, body_font_size)

        if level is not None:
            if current_text:
                sections.append({
                    "document": line["document"],
                    "source_type": "pdf",
                    "title": current_title,
                    "level": current_level,
                    "text": "\n".join(current_text),
                    "metadata": {"section_path": section_path.copy()},
                })

            current_title = text
            current_level = level
            section_path = section_path[:level - 1]
            section_path.append(text)
            current_text = []
        else:
            current_text.append(text)

    if current_text:
        sections.append({
            "document": lines[0]["document"],
            "source_type": "pdf",
            "title": current_title,
            "level": current_level,
            "text": "\n".join(current_text),
            "metadata": {"section_path": section_path.copy()},
        })

    return sections


# Usage — mirrors the DOCX flow:
#
# pdf_files = [f for f in CONFIG.raw_dir.glob("*.pdf")]
# all_pdf_units = []
# for file_path in pdf_files:
#     lines = extract_pdf_lines(file_path)
#     body_size = compute_body_font_size(lines)
#     sections = build_pdf_sections(lines, body_size)
#     all_pdf_units.extend(sections_to_structural_units(sections))
#
# Then chunk_structural_unit / chunk_dicts_to_objects work as-is — they only
# care about StructuralUnit fields, not which format it came from.